In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

In [2]:
df = pd.read_csv('/Users/loohanweiaugustine/Desktop/Final Coursework IDTA/Submission/Csv Files..../Census_Updated.csv')

/var/folders/qy/l1t4d02n3d9dr13vts802b400000gn/T/ipykernel_32526/2312603146.py:1: DtypeWarning: Columns (16) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('/Users/loohanweiaugustine/Desktop/Final Coursework IDTA/Submission/Csv Files..../Census_Updated.csv')


In [3]:
df_arm = df.copy()

In [4]:
features_to_drop = ['person_id', 'no_of_hours', 'region', 'approximated_social_grade']
df_arm = df.drop(columns=features_to_drop)

In [5]:
df_arm = df_arm[
    (~df_arm['age'].isin(['0 to 15', '75 and over'])) &
    (df_arm['economic_activity'] != 'Economically inactive: Retired') &
    (df_arm['economic_activity'] != 'Economically inactive: Student') &
    (df_arm['student'] != 'Yes')
]

In [6]:
df_arm = df_arm.fillna('Not Applicable')

In [7]:
def create_transaction(row):
    return [f'{c}={v}' for c, v in row.items() if v != 'Not Applicable']

transactions = df_arm.apply(create_transaction, axis=1).tolist()

In [8]:
te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)
df_transactions = pd.DataFrame(te_ary, columns=te.columns_)

print("Sample of transformed transactions (first 5 rows, first 10 columns):")
print(df_transactions.iloc[:5, :10])

Sample of transformed transactions (first 5 rows, first 10 columns):
   age=16 to 24  age=25 to 34  age=35 to 44  age=45 to 54  age=55 to 64  \
0         False         False         False          True         False   
1         False         False         False          True         False   
2         False         False         False         False          True   
3         False         False         False          True         False   
4         False         False          True         False         False   

   age=65 to 74  country_of_birth=Non UK  country_of_birth=UK  \
0         False                    False                 True   
1         False                    False                 True   
2         False                    False                 True   
3         False                    False                 True   
4         False                     True                False   

   economic_activity=Economically active: Employee  \
0                                  

In [9]:
min_support_val = 0.03
min_confidence_val = 0.8
min_lift_val = 1.5
max_len_val = 3

print(f"\nRunning Apriori with min_support={min_support_val}, min_confidence={min_confidence_val}, min_lift={min_lift_val}")


Running Apriori with min_support=0.03, min_confidence=0.8, min_lift=1.5


In [10]:
frequent_itemsets = apriori(
    df_transactions,
    min_support=min_support_val,
    use_colnames=True,
    max_len=max_len_val
)

In [11]:
rules = association_rules(
    frequent_itemsets,
    metric="confidence",
    min_threshold=min_confidence_val
)

rules = rules[rules['lift'] >= min_lift_val].copy()
rules.replace([np.inf, -np.inf], np.nan, inplace=True)
rules.dropna(subset=['support', 'confidence', 'lift'], inplace=True)

def cols_in(itemset):
    return {s.split('=')[0] for s in itemset}

def has_term(itemset, term):
    return any(term in s for s in itemset)

AGE_BAN = ['age=16 to 24']
for term in AGE_BAN:
    rules = rules[
        ~rules['antecedents'].apply(lambda s: has_term(s, term)) &
        ~rules['consequents'].apply(lambda s: has_term(s, term))
    ]

TRIVIAL_CONSEQ = [
    'marital_status=Married',
    'marital_status=Single',
    'marital_status=Widowed',
    'student=Yes',
    'student=No',
    'economic_activity=Economically inactive: Retired',
    'economic_activity=Economically inactive: Student',
    'family_composition=Married/same-sex civil partnership couple family'
]
rules = rules[~rules['consequents'].apply(lambda s: any(has_term(s, t) for t in TRIVIAL_CONSEQ))]

rules = rules[
    rules.apply(lambda r: len(cols_in(r['antecedents']).union(cols_in(r['consequents']))) > 1, axis=1)
]

rules = rules.sort_values(by=['lift', 'confidence', 'support'],
                          ascending=[False, False, False]).reset_index(drop=True)

print(f"\nRules after excluding 'age=16 to 24', trivial consequents, and enforcing multi-column: {len(rules)}")
print("\nTop 5 Association Rules (multi-column, non-trivial):")
for i, r in rules.head(5).iterrows():
    ant = ', '.join(sorted(list(r['antecedents'])))
    con = ', '.join(sorted(list(r['consequents'])))
    print(f"Rule {i+1}: IF {{{ant}}} THEN {{{con}}}")
    print(f"   Support: {r['support']:.3f}, Confidence: {r['confidence']:.3f}, Lift: {r['lift']:.3f}")
    print("-" * 50)


Rules after excluding 'age=16 to 24', trivial consequents, and enforcing multi-column: 85

Top 5 Association Rules (multi-column, non-trivial):
Rule 1: IF {ethnic_group=Asian and Asian British, marital_status=Married or in a registered same-sex civil partnership} THEN {country_of_birth=Non UK}
   Support: 0.042, Confidence: 0.814, Lift: 4.940
--------------------------------------------------
Rule 2: IF {industry=Construction, occupation=Skilled Trades Occupations} THEN {sex=Male}
   Support: 0.038, Confidence: 0.978, Lift: 1.922
--------------------------------------------------
Rule 3: IF {economic_activity=Economically inactive: Looking after home/family, marital_status=Married or in a registered same-sex civil partnership} THEN {sex=Female}
   Support: 0.032, Confidence: 0.920, Lift: 1.873
--------------------------------------------------
Rule 4: IF {economic_activity=Economically inactive: Looking after home/family, family_composition=Married/same-sex civil partnership couple fa

/opt/anaconda3/lib/python3.13/site-packages/mlxtend/frequent_patterns/association_rules.py:186: RuntimeWarning: invalid value encountered in divide
  cert_metric = np.where(certainty_denom == 0, 0, certainty_num / certainty_denom)
